# Escha `w0` baseline probe (single run)

Captures `torch.ops.escha.escham_reconstruct(all_zeros_code, in_f, out_f, K, True, False)` for every `(in_f, out_f, K)` combo used by Escha-W2's MoE experts and emits a base64-encoded `.npz` blob.

**Runtime**: T4 GPU, ~30 sec end-to-end.
**Action after `Runtime → Run all`**: scrape the base64 between the `BASELINE_V2_NPZ_BASE64_BEGIN/END` markers from the last cell's output.

In [ ]:
# Cell 1 — install pinned torch + hf tooling
import subprocess

def _run(cmd, check=True):
    print(f'$ {cmd}', flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise SystemExit(f'command failed: {cmd}')

_run('pip install -q torch==2.9.* --index-url https://download.pytorch.org/whl/cu128')
_run("pip install -q numpy safetensors 'huggingface_hub[cli]' hf_transfer")

In [ ]:
# Cell 2 — pull escha wheel + install (no-deps)
import os, subprocess

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

def _run(cmd, check=True):
    print(f'$ {cmd}', flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise SystemExit(f'command failed: {cmd}')

_run('mkdir -p /escha')
_run("hf download EschaLabs/escha-runtime-qwen3moe --include 'sglang/*' --local-dir /escha")
_run('pip install -q --no-deps /escha/sglang/escha-*.whl')

In [ ]:
# Cell 3 — import + GPU smoke check
import numpy as np
import torch
import escha  # registers torch.ops.escha.*

assert torch.cuda.is_available(), 'CUDA not available — Colab runtime must be a GPU.'
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'device:', torch.cuda.get_device_name(0))

op = torch.ops.escha.escham_reconstruct
device = 'cuda'

In [ ]:
# Cell 4 — probe every (in_f, out_f, K) shape used by Escha-W2 MoE
import time

SHAPES = [
    # (name, in_f, out_f, K, cshape)
    ('gate_up_proj',   2048, 1024, 2, (128,  64, 32)),  # all 40 layers × 256 experts
    ('down_proj',       512, 2048, 3, ( 32, 128, 48)),  # all 40 layers × 256 experts
    # sanity-check tiles vs docs/escha_op_signature.md §2
    ('min_K2_128x128',  128,  128, 2, (  8,   8, 32)),
    ('min_K3_128x128',  128,  128, 3, (  8,   8, 48)),
]

baselines = {}
meta = {}
t0 = time.time()
for name, in_f, out_f, K, cshape in SHAPES:
    p0 = torch.zeros(cshape, dtype=torch.int16, device=device)
    tic = time.time()
    w0 = op(p0, in_f, out_f, K, True, False)
    torch.cuda.synchronize()
    dt = time.time() - tic
    w0_np = w0.detach().cpu().numpy().astype(np.float16)
    key = f'{name}__in{in_f}_out{out_f}_K{K}'
    baselines[key] = w0_np
    meta[key] = {
        'in_f': in_f, 'out_f': out_f, 'K': K,
        'cshape': list(cshape),
        'out_shape': list(w0_np.shape),
        'out_dtype': str(w0_np.dtype),
        'l2_norm': float(np.linalg.norm(w0_np.astype(np.float32))),
        'abs_max': float(np.abs(w0_np.astype(np.float32)).max()),
        'nnz_frac': float((w0_np != 0).mean()),
        'op_seconds': dt,
    }
    print(f'  {key}: shape={w0_np.shape} '
          f"|w0|_2={meta[key]['l2_norm']:.3e} "
          f"|w0|_inf={meta[key]['abs_max']:.3e} "
          f"nnz_frac={meta[key]['nnz_frac']:.3f} ({dt*1000:.1f} ms)", flush=True)

print(f'total probe time: {time.time()-t0:.2f}s')

In [ ]:
# Cell 5 — save npz + emit base64 (scrape between BEGIN/END markers)
import base64, json, os

out_dir = '/content/escha_baseline_v2'
os.makedirs(out_dir, exist_ok=True)
npz_path = f'{out_dir}/baseline_v2.npz'
meta_path = f'{out_dir}/baseline_v2.meta.json'

save_kwargs = dict(baselines)
save_kwargs['_meta_json'] = np.frombuffer(
    json.dumps({
        'shapes': meta,
        'wheel': 'EschaLabs/escha-runtime-qwen3moe',
        'torch': torch.__version__,
    }, indent=2).encode('utf-8'),
    dtype=np.uint8,
)
np.savez_compressed(npz_path, **save_kwargs)
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

size = os.path.getsize(npz_path)
print(f'saved: {npz_path} ({size/1024:.1f} KiB)')
print(f'saved: {meta_path}')

with open(npz_path, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode('ascii')

print('\n=========== BASELINE_V2_NPZ_BASE64_BEGIN ===========')
CHUNK = 76
for i in range(0, len(b64), CHUNK):
    print(b64[i:i+CHUNK])
print('=========== BASELINE_V2_NPZ_BASE64_END =============')
print(f'base64 length: {len(b64)} chars ({size} bytes)')